# EDA Template — 재무비율 기반 상폐 예측 데이터셋

이 노트북은 `clean_data` 도착 시 즉시 실행할 수 있는 탐색적 데이터 분석 템플릿입니다.

## 1. 설정 & 데이터 로드

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 프로젝트 루트를 path에 추가
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis.utils import (
    analyze_single_feature,
    classify_columns,
    compare_group_stats_by_label,
    detect_outliers_iqr,
    get_high_corr_pairs,
    high_missing_columns,
    load_csv,
    missing_summary,
    plot_boxplot_by_label,
    plot_correlation_heatmap,
    plot_histogram,
    plot_missing_heatmap,
    summarize_dataframe,
)

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

sns.set_theme(style="whitegrid")

In [ ]:
# 데이터 경로 설정 (매크로 포함 / 미포함 선택)
DATA_PATH = PROJECT_ROOT / "preprocess" / "data" / "output" / "clean_data_no_macro.csv"
# DATA_PATH = PROJECT_ROOT / "preprocess" / "data" / "output" / "clean_data.csv"

df = load_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

## 2. 컬럼 분류

In [ ]:
cols = classify_columns(df)

for group, col_list in cols.items():
    print(f"\n[{group}] ({len(col_list)}개)")
    print(col_list)

## 3. 기본 통계

In [ ]:
summarize_dataframe(df)

In [ ]:
df[cols["ratio"]].describe().T

## 4. 결측 분석

In [ ]:
missing_summary(df)

In [ ]:
high_miss = high_missing_columns(df, threshold=0.3)
print(f"결측률 30% 이상 컬럼: {high_miss}")

In [ ]:
plot_missing_heatmap(df)

## 5. 라벨 분포

In [ ]:
label_counts = df["label"].value_counts()
print("Label 분포:")
print(label_counts)
print(f"\n불균형 비율 (delisted/healthy): {label_counts.get(1, 0) / label_counts.get(0, 1):.4f}")

In [ ]:
sector_label = df.groupby("gics_sector")["label"].value_counts().unstack(fill_value=0)
sector_label["delisted_ratio"] = sector_label[1] / (sector_label[0] + sector_label[1])
sector_label.sort_values("delisted_ratio", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sector_label[[0, 1]].plot(kind="bar", stacked=True, ax=ax)
ax.set_title("GICS 섹터별 Label 분포")
ax.set_xlabel("GICS Sector")
ax.set_ylabel("건수")
ax.legend(["healthy (0)", "delisted (1)"])
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. 단변량 분포

주요 비율 컬럼을 선정하여 분포와 label별 차이를 확인한다.

In [ ]:
# 분석 대상 피처 (필요에 따라 수정)
TARGET_FEATURES = [
    "부채비율",
    "유동비율",
    "자기자본비율",
    "매출액순이익률",
    "자기자본순이익률",
    "총자본영업이익률",
    "총자본회전율",
    "차입금의존도",
    "매출총이익률",
    "현금비율",
]

In [ ]:
for feat in TARGET_FEATURES:
    if feat not in df.columns:
        print(f"[SKIP] {feat} 없음")
        continue
    print(f"\n{'='*60}")
    print(f"  {feat}")
    print(f"{'='*60}")

    stats = analyze_single_feature(df, feat)
    for k, v in stats.items():
        print(f"  {k}: {v}")

    plot_histogram(df, feat)
    plot_boxplot_by_label(df, feat)

    print("\n[Label별 비교]")
    display(compare_group_stats_by_label(df, feat))

## 7. 상관관계

In [ ]:
plot_correlation_heatmap(df)

In [ ]:
high_corr = get_high_corr_pairs(df, threshold=0.8)
print(f"상관계수 절대값 ≥ 0.8 쌍: {len(high_corr)}개\n")
for col_a, col_b, corr_val in high_corr:
    print(f"  {col_a} <-> {col_b}: {corr_val}")

### 다중공선성 후보 정리

In [ ]:
if high_corr:
    drop_candidates = set()
    for col_a, col_b, _ in high_corr:
        # 결측률이 더 높은 쪽을 제거 후보로
        miss_a = df[col_a].isnull().mean()
        miss_b = df[col_b].isnull().mean()
        drop_candidates.add(col_a if miss_a >= miss_b else col_b)
    print(f"다중공선성 제거 후보 ({len(drop_candidates)}개): {sorted(drop_candidates)}")
else:
    print("다중공선성 후보 없음")

## 8. Baseline 모델 (간단 실행)

In [ ]:
from src.baseline.run_baseline import (
    evaluate,
    prepare_features,
    print_report,
    time_split,
    train_decision_tree,
    train_logistic,
)

In [ ]:
train_df, val_df, test_df = time_split(df, train_end=2021, val_end=2022)
print(f"train: {len(train_df):,}행, val: {len(val_df):,}행, test: {len(test_df):,}행")

X_train, y_train = prepare_features(train_df)
X_val, y_val = prepare_features(val_df)
X_test, y_test = prepare_features(test_df)

In [ ]:
lr_model = train_logistic(X_train, y_train)
dt_model = train_decision_tree(X_train, y_train)

results = {
    "LogisticRegression": evaluate(lr_model, X_test, y_test),
    "DecisionTree(depth=5)": evaluate(dt_model, X_test, y_test),
}

print_report(results)